In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 데이터 불러오기 
names = pd.read_csv("../Data/tpss_filtered_202301.csv",encoding='cp949')
names.head()


,기준_날짜,기준_시간대,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납
0,20230101,0,ST-105,ST-361,9,1635,True,True
1,20230101,0,ST-244,ST-1201,12,2810,True,False
2,20230101,0,ST-3003,ST-381,22,3320,True,False
3,20230101,0,ST-366,ST-2948,60,10408,True,False
4,20230101,0,ST-371,ST-2330,9,1932,True,True


In [3]:
names[(names['대여'] == False) & (names['반납'] == False)]

,기준_날짜,기준_시간대,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납


In [ ]:
# # 1. 결측치 제거
# names = names.dropna(subset=['기준_날짜'])

# # 2. float → int → str 변환 (예: 20230201.0 → 20230201)
# names['기준_날짜'] = names['기준_날짜'].astype(int)

# names.head()

,기준_날짜,기준_시간대,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납
0,20230201,0,ST-2339,ST-249,8,652,True,True
1,20230201,0,ST-2340,ST-390,16,2080,True,False
2,20230201,0,ST-366,ST-2339,11,2636,True,True
3,20230201,0,ST-446,ST-106,8,440,True,True
4,20230201,0,ST-827,ST-360,47,2810,False,True


In [ ]:
# !pip install holidays

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
from datetime import datetime
import holidays

# 기준_날짜를 datetime 형식으로 변환
names['기준_날짜'] = pd.to_datetime(names['기준_날짜'], format='%Y%m%d')

# 한국 공휴일 정보 가져오기
kr_holidays = holidays.KR(years=[2023])  # 필요한 연도 추가 가능

# 요일/공휴일 구분 함수 (3단계 분류)
def classify_day(date):
    if date in kr_holidays:
        return '공휴일'
    elif date.weekday() >= 5:
        return '주말'
    else:
        return '주중'

# 새로운 컬럼 추가
names['요일구분'] = names['기준_날짜'].apply(classify_day)

In [5]:
names

,기준_날짜,기준_시간대,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납,요일구분
0,2023-01-01,0,ST-105,ST-361,9,1635,True,True,공휴일
1,2023-01-01,0,ST-244,ST-1201,12,2810,True,False,공휴일
2,2023-01-01,0,ST-3003,ST-381,22,3320,True,False,공휴일
3,2023-01-01,0,ST-366,ST-2948,60,10408,True,False,공휴일
4,2023-01-01,0,ST-371,ST-2330,9,1932,True,True,공휴일
...,...,...,...,...,...,...,...,...,...
149054,2023-01-31,1340,ST-2408,ST-243,52,6904,False,True,주중
149055,2023-01-31,1645,ST-973,ST-366,258,29420,False,True,주중
149056,2023-01-31,1845,ST-3042,ST-482,24,4135,True,True,주중
149057,2023-01-31,1850,ST-3210,ST-2338,227,13447,False,True,주중


In [6]:
# 성동구 대여소 ID 불러오기
id = pd.read_csv("../Data/따릉이_성동구_대여소.csv", encoding='utf-8')

In [7]:
id.head()

,대여소_ID,주소1,주소2,위도,경도
0,ST-967,서울특별시 성동구 자동차시장길 49,중랑물재생센터,37.558304,127.058060
1,ST-506,서울특별시 성동구 아차산로 100,성수동2가 315-105,37.544159,127.056656
2,ST-482,서울특별시 성동구 성수동2가 276-5,KEB하나은행 성수중앙지점,37.545418,127.052643
3,ST-48,서울특별시 성동구 자동차시장 3길 64 (물재생센터),운영센터,37.557598,127.065308
4,ST-447,서울특별시 성동구 마장로42길 14,마장동 주민센터,37.565941,127.045395


In [14]:
# 성동구 내 대여소에서 대여된 건수
rent_count = names[names['시작_대여소ID'].isin(id)]['시작_대여소ID'].value_counts().reset_index()
rent_count.columns = ['대여소_ID', '대여_건수']

# 성동구 내 대여소에서 반납된 건수
return_count = names[names['종료_대여소ID'].isin(id)]['종료_대여소ID'].value_counts().reset_index()
return_count.columns = ['대여소_ID', '반납_건수']

In [15]:
# 두 집계 테이블 병합
merged = pd.merge(rent_count, return_count, on='대여소_ID', how='outer').fillna(0)

# 정수형으로 변환
merged['대여_건수'] = merged['대여_건수'].astype(int)
merged['반납_건수'] = merged['반납_건수'].astype(int)

# 차이 계산
merged['반납 - 대여'] = merged['반납_건수'] - merged['대여_건수']

# 반납이 많은 순 정렬
merged_sorted = merged.sort_values(by='반납 - 대여', ascending=False)

In [16]:
# 상위 10개 확인
print(merged_sorted.head(10))

# 또는 전체 보기
pd.set_option('display.max_rows', None)  # 모든 행 보기
pd.set_option('display.max_columns', None)  # 모든 열 보기
print(merged_sorted)

Empty DataFrame
Columns: [대여_건수, 대여소_ID, 반납_건수, 반납 - 대여]
Index: []
Empty DataFrame
Columns: [대여_건수, 대여소_ID, 반납_건수, 반납 - 대여]
Index: []


In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib

plt.figure(figsize=(12, 6))
merged_sorted.set_index('대여소_ID')['반납 - 대여'].plot(kind='bar', color='skyblue')
plt.title('성동구 2023년 1월 대여소별 반납 - 대여 건수')
plt.ylabel('반납 - 대여 건수')
plt.xlabel('대여소 ID')
plt.xticks(rotation=90)
plt.grid(True)
plt.tight_layout()
plt.show()

NameError: name 'merged_sorted' is not defined

<Figure size 1200x600 with 0 Axes>

In [85]:
# 병합할 파일 리스트 생성
files = [f"../Data/tpss_filtered_2023{str(i).zfill(2)}.csv" for i in range(1, 13)]

# 병합된 데이터를 담을 리스트
df_list = []

# 각 파일을 읽어서 리스트에 추가
for file in files:
    df = pd.read_csv(file, encoding='cp949')  # 인코딩은 상황에 따라 'utf-8'로 바꿔도 됨
    df_list.append(df)

# 모든 데이터프레임 병합
merged_df = pd.concat(df_list, ignore_index=True)

# CSV로 저장
merged_df.to_csv("../Data/tpss_filtered_2023_merged.csv", index=False, encoding='utf-8-sig')

print("병합 완료: tpss_filtered_2023_merged.csv")

병합 완료: tpss_filtered_2023_merged.csv


In [8]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
import pandas as pd
from datetime import datetime
import holidays


# 데이터 불러오기 
df_2023 = pd.read_csv("../Data/tpss_filtered_2023_merged.csv",encoding='utf-8-sig')
df_2023.head()

,기준_날짜,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납
0,20230101,ST-105,ST-361,9,1635,True,True
1,20230101,ST-244,ST-1201,12,2810,True,False
2,20230101,ST-3003,ST-381,22,3320,True,False
3,20230101,ST-366,ST-2948,60,10408,True,False
4,20230101,ST-371,ST-2330,9,1932,True,True


In [9]:
df_2023[(df_2023['대여'] == False) & (df_2023['반납'] == False)]

,기준_날짜,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납


In [10]:
# 기준_날짜를 datetime 형식으로 변환
df_2023['기준_날짜'] = pd.to_datetime(df_2023['기준_날짜'], format='%Y%m%d')

# 한국 공휴일 정보 가져오기
kr_holidays = holidays.KR(years=[2023])  # 연도 조정 필요 시 리스트로 전달

# 요일/공휴일 구분 함수
def classify_day(date):
    if date in kr_holidays or date.weekday() >= 5:
        return '주말_공휴일'  # 토/일 또는 공휴일
    else:
        return '평일'

# 새로운 컬럼 추가
df_2023['요일구분'] = df_2023['기준_날짜'].apply(classify_day)

df_2023.head()

,기준_날짜,시작_대여소ID,종료_대여소ID,전체이용시간(분),전체이용거리(m),대여,반납,요일구분
0,2023-01-01,ST-105,ST-361,9,1635,True,True,주말_공휴일
1,2023-01-01,ST-244,ST-1201,12,2810,True,False,주말_공휴일
2,2023-01-01,ST-3003,ST-381,22,3320,True,False,주말_공휴일
3,2023-01-01,ST-366,ST-2948,60,10408,True,False,주말_공휴일
4,2023-01-01,ST-371,ST-2330,9,1932,True,True,주말_공휴일


In [16]:
# 데이터 불러오기 
df_2023_air = pd.read_csv("../Data/서울시 대기질 자료 제공_2023.csv",encoding='cp949')
df_2023_air.head()

,일시,구분,미세먼지(PM10),초미세먼지(PM25)
0,2023-12-31 23:00,평균,31.0,25.0
1,2023-12-31 23:00,강남구,33.0,30.0
2,2023-12-31 23:00,강동구,32.0,20.0
3,2023-12-31 23:00,강북구,14.0,12.0
4,2023-12-31 23:00,강서구,36.0,24.0


In [31]:
# 일시 컬럼을 datetime 형식으로 변환
df_2023_air['일시'] = pd.to_datetime(df_2023_air['일시'])

# yyyy-mm-dd 형태로 변환 (시간 제거)
df_2023_air['일시'] = df_2023_air['일시'].dt.date.astype(str)

# 성동구 데이터만 필터링
seongdong_df = df_2023_air[df_2023_air['구분'] == '성동구']

# 결과 출력
seongdong_df.head()

,일시,구분,미세먼지(PM10),초미세먼지(PM25)
16,2023-12-31,성동구,30.0,26.0
42,2023-12-31,성동구,33.0,28.0
68,2023-12-31,성동구,31.0,25.0
94,2023-12-31,성동구,22.0,18.0
120,2023-12-31,성동구,19.0,16.0


In [33]:
grouped_air = seongdong_df.groupby('일시').agg({
    '미세먼지(PM10)': 'mean',
    '초미세먼지(PM25)': 'mean'
}).reset_index()

In [34]:
# 등급 순서를 숫자로 매핑
grade_rank = {
    '좋음': 1,
    '보통': 2,
    '나쁨': 3,
    '매우나쁨': 4
}

# 등급 판별 함수
def get_air_grade(value, pollutant='PM10'):
    if pollutant == 'PM10':
        if value <= 30:
            return '좋음'
        elif value <= 80:
            return '보통'
        elif value <= 150:
            return '나쁨'
        else:
            return '매우나쁨'
    elif pollutant == 'PM25':
        if value <= 15:
            return '좋음'
        elif value <= 35:
            return '보통'
        elif value <= 75:
            return '나쁨'
        else:
            return '매우나쁨'

# 등급 계산
grouped_air['PM10_등급'] = grouped_air['미세먼지(PM10)'].apply(lambda x: get_air_grade(x, 'PM10'))
grouped_air['PM25_등급'] = grouped_air['초미세먼지(PM25)'].apply(lambda x: get_air_grade(x, 'PM25'))

# 종합 등급: 더 나쁜 쪽 기준으로 선택
def get_overall_grade(row):
    pm10_rank = grade_rank[row['PM10_등급']]
    pm25_rank = grade_rank[row['PM25_등급']]
    worst_rank = max(pm10_rank, pm25_rank)
    return [k for k, v in grade_rank.items() if v == worst_rank][0]

grouped_air['대기등급'] = grouped_air.apply(get_overall_grade, axis=1)



grouped_air.drop(columns=['미세먼지(PM10)', '초미세먼지(PM25)', 'PM10_등급', 'PM25_등급'], inplace=True)
grouped_air.head()

,일시,대기등급
0,2023-01-01,보통
1,2023-01-02,좋음
2,2023-01-03,좋음
3,2023-01-04,보통
4,2023-01-05,보통


In [35]:
grouped_air.rename(columns={"일시": "기준_날짜"}, inplace=True)
grouped_air.tail()

,기준_날짜,대기등급
360,2023-12-27,나쁨
361,2023-12-28,나쁨
362,2023-12-29,보통
363,2023-12-30,보통
364,2023-12-31,보통


In [38]:
# df_2023이 전체 데이터프레임이라고 가정
melted = pd.melt(
    df_2023,
    id_vars=['기준_날짜', '전체이용시간(분)', '전체이용거리(m)', '요일구분'],
    value_vars=['시작_대여소ID', '종료_대여소ID'],
    var_name='대여_구분',
    value_name='대여소ID'
)


# 원하는 순서로 열 재배치
melted = melted[[
    '대여소ID',
    '기준_날짜',
    '대여_구분',
    '전체이용시간(분)',
    '전체이용거리(m)',
    '요일구분'
]]

melted.tail()

,대여소ID,기준_날짜,대여_구분,전체이용시간(분),전체이용거리(m),요일구분
7716437,ST-145,2023-12-31,종료_대여소ID,30,6170,주말_공휴일
7716438,ST-274,2023-12-31,종료_대여소ID,20,2780,주말_공휴일
7716439,ST-103,2023-12-31,종료_대여소ID,4,520,주말_공휴일
7716440,ST-176,2023-12-31,종료_대여소ID,15,1848,주말_공휴일
7716441,ST-410,2023-12-31,종료_대여소ID,5,1280,주말_공휴일


In [41]:
melted['기준_날짜'] = pd.to_datetime(melted['기준_날짜'])
grouped_air['기준_날짜'] = pd.to_datetime(grouped_air['기준_날짜'])

merged_df = pd.merge(melted, grouped_air, on='기준_날짜', how='left')
merged_df.head()

,대여소ID,기준_날짜,대여_구분,전체이용시간(분),전체이용거리(m),요일구분,대기등급
0,ST-105,2023-01-01,시작_대여소ID,9,1635,주말_공휴일,보통
1,ST-244,2023-01-01,시작_대여소ID,12,2810,주말_공휴일,보통
2,ST-3003,2023-01-01,시작_대여소ID,22,3320,주말_공휴일,보통
3,ST-366,2023-01-01,시작_대여소ID,60,10408,주말_공휴일,보통
4,ST-371,2023-01-01,시작_대여소ID,9,1932,주말_공휴일,보통


In [12]:
# 1. 대여건수
rental_counts = (
    melted[melted['대여_구분'] == '시작_대여소ID']
    .groupby(['기준_날짜', '대여소ID'])
    .size()
    .reset_index(name='대여건수')
)

# 2. 반납건수
return_counts = (
    melted[melted['대여_구분'] == '종료_대여소ID']
    .groupby(['기준_날짜', '대여소ID'])
    .size()
    .reset_index(name='반납건수')
)

# 3. 병합
summary = pd.merge(rental_counts, return_counts, on=['기준_날짜', '대여소ID'], how='outer').fillna(0)

# 4. 반납 - 대여 컬럼 추가
summary['반납-대여'] = summary['반납건수'] - summary['대여건수']

# 5. 정수형으로 변환 (소수점 방지)
summary[['대여건수', '반납건수', '반납-대여']] = summary[['대여건수', '반납건수', '반납-대여']].astype(int)

# 결과 확인
print(summary[summary['대여소ID'] == 'ST-105'])

            기준_날짜   대여소ID  대여건수  반납건수  반납-대여
3      2023-01-01  ST-105    51    38    -13
296    2023-01-02  ST-105    95    61    -34
645    2023-01-03  ST-105    80    66    -14
1006   2023-01-04  ST-105    72    66     -6
1394   2023-01-05  ST-105   114    98    -16
...           ...     ...   ...   ...    ...
177655 2023-12-27  ST-105    87    90      3
178083 2023-12-28  ST-105    82    66    -16
178514 2023-12-29  ST-105    94   100      6
178920 2023-12-30  ST-105    12    10     -2
179108 2023-12-31  ST-105    27    30      3

[339 rows x 5 columns]


In [13]:
summary.head()

,기준_날짜,대여소ID,대여건수,반납건수,반납-대여
0,2023-01-01,ST-1015,2,0,-2
1,2023-01-01,ST-102,4,4,0
2,2023-01-01,ST-103,20,9,-11
3,2023-01-01,ST-105,51,38,-13
4,2023-01-01,ST-106,108,108,0


In [ ]:
summary.to_csv("../Data/summary_2023.csv", index=False, encoding='utf-8-sig')